# Interim: train the 8 CREMA-D fusion variants

Assumes `notebooks/interim_01_data_prep.ipynb` has run to completion and `data/crema_d/annotations/{train,val,test}.txt` + `cache/features/crema_d/...` exist.

Variants in run order:
1. `visual_only`  (unimodal baseline -- also used by F0)
2. `audio_only`   (unimodal baseline -- also used by F0)
3. `f1_concat`    (early concat)
4. `f2_blend`     (learned per-feature blend)
5. `f3_gate`      (task-specific gating)
6. `f4_xattn`     (bidirectional cross-attention, windowed)
7. `f5_lmf`       (low-rank multimodal fusion)
8. `f0_grid`      (post-hoc grid blend over the two trained unimodals)

Output: one `results/interim/crema_<variant>/` directory per run, each with `best.pt`, `log.csv`, `metrics.md`. A combined summary table is written to `results/interim/summary_crema.md`.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("cwd:", Path.cwd())

from omegaconf import OmegaConf
from src.train_fusion import train_from_config
from src.eval_fusion import evaluate_variant, evaluate_f0_grid

CONFIGS = ROOT / "configs" / "interim"
RESULTS = ROOT / "results" / "interim"
RESULTS.mkdir(parents=True, exist_ok=True)

# Order matters: F0 depends on trained visual_only + audio_only checkpoints.
TRAINABLE = [
    "crema_visual_only",
    "crema_audio_only",
    "crema_f1_concat",
    "crema_f2_blend",
    "crema_f3_gate",
    "crema_f4_xattn",
    "crema_f5_lmf",
]

cwd: C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code


## Train each trainable variant

Re-running a completed variant skips training only if you comment it out -- `train_from_config` overwrites `best.pt`. Use `epochs: 0` in the YAML if you need to regenerate the metrics file without retraining.

In [2]:
checkpoints = {
    name: RESULTS / name / "best.pt"
    for name in TRAINABLE
    if (RESULTS / name / "best.pt").exists()
}
print("found:", list(checkpoints))

found: ['crema_visual_only', 'crema_audio_only', 'crema_f1_concat', 'crema_f2_blend', 'crema_f3_gate', 'crema_f4_xattn', 'crema_f5_lmf']


In [6]:
checkpoints = {}
for name in TRAINABLE:
    cfg = OmegaConf.load(CONFIGS / f"{name}.yaml")
    print(f"\n=== training {name} ===")
    ck = train_from_config(cfg)
    checkpoints[name] = ck
print("\ntrained:", {k: str(v) for k, v in checkpoints.items()})


=== training crema_visual_only ===
[fusion=visual_only] trainable params = 179,706
[visual_only] ep   1  train=1.9216  val=1.8576  best=1.8576  *
[visual_only] ep   2  train=1.5946  val=1.8022  best=1.8022  *
[visual_only] ep   3  train=1.5065  val=1.7855  best=1.7855  *
[visual_only] ep   4  train=1.4506  val=1.7817  best=1.7817  *
[visual_only] ep   5  train=1.4083  val=1.7760  best=1.7760  *
[visual_only] ep   6  train=1.3756  val=1.7756  best=1.7756  *
[visual_only] ep   7  train=1.3487  val=1.7751  best=1.7751  *
[visual_only] ep   8  train=1.3267  val=1.7813  best=1.7751
[visual_only] ep   9  train=1.3072  val=1.7876  best=1.7751
[visual_only] ep  10  train=1.2901  val=1.7859  best=1.7751
[visual_only] ep  11  train=1.2750  val=1.7872  best=1.7751
[visual_only] ep  12  train=1.2618  val=1.7967  best=1.7751
[visual_only] ep  13  train=1.2487  val=1.7977  best=1.7751
[visual_only] ep  14  train=1.2379  val=1.7994  best=1.7751
[visual_only] ep  15  train=1.2269  val=1.8090  best=1.

## Evaluate each variant

Reads `best.pt` + its config, rebuilds the val loader, writes `metrics.md`, and returns a dict with CCC_V, CCC_A, F1_EXPR, F1_AU, P_MTL.

In [3]:
metrics = {}
for name, ck in checkpoints.items():
    print(f"\n=== evaluating {name} ===")
    m = evaluate_variant(
        checkpoint=ck,
        config_path=CONFIGS / f"{name}.yaml",
    )
    metrics[name] = m


=== evaluating crema_visual_only ===


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote results\interim\crema_visual_only\metrics.md
               ccc_V = 0.6821
               ccc_A = 0.2989
              CCC_VA = 0.4905
       F1_EXPR_macro = 0.3799
            ACC_EXPR = 0.5097
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 0.8704
          P_MTL_best = 0.8704
             variant = visual_only
    trainable_params = 179706.0000

=== evaluating crema_audio_only ===


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote results\interim\crema_audio_only\metrics.md
               ccc_V = 0.2688
               ccc_A = 0.4645
              CCC_VA = 0.3667
       F1_EXPR_macro = 0.3021
            ACC_EXPR = 0.4175
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 0.6687
          P_MTL_best = 0.6687
             variant = audio_only
    trainable_params = 142998.0000

=== evaluating crema_f1_concat ===


...

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote results\interim\crema_f3_gate\metrics.md
               ccc_V = 0.7352
               ccc_A = 0.5141
              CCC_VA = 0.6246
       F1_EXPR_macro = 0.4270
            ACC_EXPR = 0.5697
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 1.0517
          P_MTL_best = 1.0517
             variant = f3_gate
    trainable_params = 624168.0000

=== evaluating crema_f4_xattn ===


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[stage3] wrote results\interim\crema_f4_xattn\metrics.md
               ccc_V = 0.8333
               ccc_A = 0.5390
              CCC_VA = 0.6861
       F1_EXPR_macro = 0.4886
            ACC_EXPR = 0.6524
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 1.1747
          P_MTL_best = 1.1747
             variant = f4_xattn
    trainable_params = 1720470.0000

=== evaluating crema_f5_lmf ===
[stage3] wrote results\interim\crema_f5_lmf\metrics.md
               ccc_V = 0.7064
               ccc_A = 0.4570
              CCC_VA = 0.5817
       F1_EXPR_macro = 0.4422
            ACC_EXPR = 0.5943
           F1_AU@0.5 = 0.0000
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
           P_MTL@0.5 = 1.0239
          P_MTL_best = 1.0239
             variant = f5_lmf
    trainable_params = 597914.0000


c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## F0 post-hoc grid blend

Uses the two trained unimodal checkpoints and grid-searches a scalar weight per task. No training, so we call `evaluate_f0_grid` directly.

In [5]:
import importlib                                                                                                                                  
import src.fusion.f0_grid, src.eval_fusion                                                                                                        
importlib.reload(src.fusion.f0_grid)
importlib.reload(src.eval_fusion)
from src.eval_fusion import evaluate_variant, evaluate_f0_grid

In [6]:
m_f0 = evaluate_f0_grid(
    visual_checkpoint=checkpoints["crema_visual_only"],
    audio_checkpoint=checkpoints["crema_audio_only"],
    config_path=CONFIGS / "crema_f0_grid.yaml",
)
metrics["crema_f0_grid"] = m_f0

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c

[stage3] wrote results\interim\crema_f0_grid\metrics_f0.md
               ccc_V = 0.6325
               ccc_A = 0.4222
              CCC_VA = 0.5274
       F1_EXPR_macro = 0.4519
          F1_AU_best = 0.0000
           t_AU_best = 0.5000
               P_MTL = 1.5066
              w_expr = 0.3000
                w_va = 0.6000
                w_au = 0.0000
             variant = f0_grid


## Summary table

Combined Markdown + pandas DataFrame for the report. Columns match the existing AffWild2 results table so we can drop the values into the thesis unchanged.

In [9]:
import pandas as pd

COLS = ["variant", "ccc_V", "ccc_A", "CCC_VA",
        "F1_EXPR_macro", "F1_AU@0.5", "F1_AU_best",
        "P_MTL@0.5", "trainable_params"]

def row(name, m):
    r = {"run": name}
    for c in COLS:
        r[c] = m.get(c, "")
    return r

df = pd.DataFrame([row(n, m) for n, m in metrics.items()])

summary_path = RESULTS / "summary_crema.md"
with summary_path.open("w", encoding="utf-8") as fh:
    fh.write("# CREMA-D interim results\n\n")
    fh.write(df.to_markdown(index=False, floatfmt=".4f"))
    fh.write("\n")

print("wrote", summary_path)
df

wrote C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\results\interim\summary_crema.md


,run,variant,ccc_V,ccc_A,CCC_VA,F1_EXPR_macro,F1_AU@0.5,F1_AU_best,P_MTL@0.5,trainable_params
0,crema_visual_only,visual_only,0.682083,0.298906,0.490495,0.379893,0.0,0.0,0.870387,179706
1,crema_audio_only,audio_only,0.268824,0.464530,0.366677,0.302064,0.0,0.0,0.668741,142998
2,crema_f1_concat,f1_concat,0.740063,0.452515,0.596289,0.438914,0.0,0.0,1.035203,995690
3,crema_f2_blend,f2_blend,0.695278,0.436848,0.566063,0.431998,0.0,0.0,0.99806,322708
4,crema_f3_gate,f3_gate,0.735184,0.514094,0.624639,0.427045,0.0,0.0,1.051685,624168
5,crema_f4_xattn,f4_xattn,0.833264,0.539012,0.686138,0.488603,0.0,0.0,1.174741,1720470
6,crema_f5_lmf,f5_lmf,0.706425,0.456956,0.581691,0.442167,0.0,0.0,1.023858,597914
7,crema_f0_grid,f0_grid,0.632493,0.422222,0.527358,0.451874,,0.0,,


In [12]:
import time                                                                                                                                         
import torch                                                                                                                                        
from omegaconf import OmegaConf                                                                                                                     
from src.train_fusion import build_model, FusionConfig                                                                                              
                
def bench_one(name: str, ckpt_path, n_iter: int = 500) -> float:
    cfg = OmegaConf.load(CONFIGS / f"{name}.yaml")
    fus_cfg = FusionConfig(
        v_dim=int(cfg.visual.feature_dim),
        scores_dim=int(cfg.visual.scores_dim),
        a_dim=int(cfg.audio.feature_dim),
        num_expr=int(cfg.data.num_expr),
        num_aus=int(cfg.data.num_aus),
        hidden=int(cfg.fusion.get("hidden", 384)),
        dropout=float(cfg.fusion.get("dropout", 0.3)),
        au_hidden=int(cfg.head.au_hidden),
    )
    model = build_model(cfg.fusion.variant, fus_cfg).to("cpu").eval()
    state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(state["state_dict"])
    v = torch.zeros(1, fus_cfg.v_dim)
    s = torch.zeros(1, fus_cfg.scores_dim)
    a = torch.zeros(1, fus_cfg.a_dim)
    with torch.no_grad():
        for _ in range(50):                # warmup
            model(v, s, a)
        t0 = time.perf_counter()
        for _ in range(n_iter):
            model(v, s, a)
    return (time.perf_counter() - t0) * 1000.0 / n_iter

lat = {}
for name, ck in checkpoints.items():
    ms = bench_one(name, ck)
    lat[name] = ms
    print(f"{name:28s}  {ms:6.2f} ms/frame")

crema_visual_only               0.13 ms/frame
crema_audio_only                0.12 ms/frame
crema_f1_concat                 0.51 ms/frame
crema_f2_blend                  0.35 ms/frame
crema_f3_gate                   0.61 ms/frame
crema_f4_xattn                  1.82 ms/frame
crema_f5_lmf                    0.96 ms/frame
